In [ ]:
import random
import html as html_utils
import gradio as gr
from config import *
from interview_bot import create_session_state, start_session, handle_message
from grader import grade_report, preload_language_tool
from session_logger import save_session

In [ ]:
def handle_start(selected_crime_types):
    if not selected_crime_types:
        return (
            gr.update(),                    # dispatch bar
            [],                             # chatbot
            create_session_state(),         # state
            gr.update(visible=True),        # crime type selector
            gr.update(visible=True),        # start button
            gr.update(visible=False),       # session info
            gr.update(visible=False),       # end session button
            gr.update(selected="interview"),# tabs
            gr.update(visible=True),        # interview tab
            gr.update(visible=False),       # report tab
            gr.update(visible=False),       # grading tab
            gr.update(interactive=True),    # message input
            gr.update(interactive=True),    # send button
            gr.update(interactive=True),    # end interview button
            "Please select at least one crime type before starting."
        )

    crime_type = random.choice(selected_crime_types)
    preload_language_tool()
    state, dispatch_line = start_session(crime_type)

    return (
        gr.update(value=f"**Dispatch:** {dispatch_line}", visible=True),  # dispatch bar
        [],                                                                 # chatbot
        state,                                                              # state
        gr.update(visible=False),                                           # crime type selector
        gr.update(visible=False),                                           # start button
        gr.update(value=f"**Interviewee:** {state['scenario']['interviewee_name']}\n**Role:** Caller\n**Crime type:** {state['scenario']['crime_type']}", visible=True),  # session info
        gr.update(visible=True),                                            # end session button
        gr.update(selected="interview"),                                    # tabs
        gr.update(visible=True),                                            # interview tab
        gr.update(visible=False),                                           # report tab
        gr.update(visible=False),                                           # grading tab
        gr.update(interactive=True),                                        # message input
        gr.update(interactive=True),                                        # send button
        gr.update(interactive=True),                                        # end interview button
        ""                                                                  # error message
    )

In [ ]:
def handle_chat(officer_message, history, state):
    if not officer_message or not officer_message.strip():
        return history, "", state

    if not state or not state.get("scenario") or not state.get("checklist") or not state.get("coverage_flags"):
        return history or [], "", state or create_session_state()

    if state.get("phase") != "interview":
        return history or [], "", state

    response, history, state = handle_message(officer_message, history, state)

    return history, "", state

In [ ]:
def handle_request_end_interview(state):
    if not state or state.get("phase") != "interview":
        return (
            state or create_session_state(),
            gr.update(visible=False),
            "You must start an interview before ending it."
        )

    if not state.get("history"):
        return (
            state,
            gr.update(visible=False),
            "You must ask at least one question before ending the interview."
        )

    return (
        state,
        gr.update(visible=True),
        ""
    )

In [ ]:
def handle_cancel_end_interview():
    return gr.update(visible=False), ""

In [ ]:
def handle_confirm_end_interview(state):
    state["phase"] = "report"

    return (
        state,
        gr.update(selected="report"),
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(visible=False),
        gr.update(interactive=False),
        gr.update(interactive=False),
        gr.update(interactive=False),
        gr.update(visible=False),
        gr.update(interactive=True),
        gr.update(interactive=True),
        ""
    )

In [ ]:
def handle_request_submit_report(narrative, state):
    if not narrative or not narrative.strip():
        return (
            gr.update(visible=False),
            "Please write your report before submitting."
        )

    if not state or state.get("phase") != "report":
        return (
            gr.update(visible=False),
            "You must complete the interview before submitting a report."
        )

    return (
        gr.update(visible=True),
        ""
    )

In [ ]:
def handle_cancel_submit_report():
    return gr.update(visible=False), ""

In [ ]:
def submit_report_response(
    state,
    pdf_download_update=None,
    tabs_update=None,
    grading_display_update=None,
    error_message_update="",
    interview_tab_update=None,
    report_tab_update=None,
    grading_tab_update=None,
    message_input_update=None,
    send_btn_update=None,
    end_interview_btn_update=None,
    narrative_input_update=None,
    submit_report_btn_update=None,
    submit_report_confirm_update=None
):
    return (
        state,
        pdf_download_update or gr.update(),
        tabs_update or gr.update(),
        grading_display_update or gr.update(),
        error_message_update,
        interview_tab_update or gr.update(),
        report_tab_update or gr.update(),
        grading_tab_update or gr.update(),
        message_input_update or gr.update(),
        send_btn_update or gr.update(),
        end_interview_btn_update or gr.update(),
        narrative_input_update or gr.update(),
        submit_report_btn_update or gr.update(),
        submit_report_confirm_update or gr.update()
    )

In [ ]:
def handle_submit_report(narrative, state, progress=gr.Progress()):
    if not narrative or not narrative.strip():
        return submit_report_response(
            state,
            error_message_update="Please write your report before submitting."
        )

    if not state or not state.get("scenario") or not state.get("checklist") or not state.get("coverage_flags"):
        return submit_report_response(
            state or create_session_state(),
            tabs_update=gr.update(selected="report"),
            error_message_update="Please start a session before submitting a report.",
            submit_report_confirm_update=gr.update(visible=False)
        )

    if state.get("phase") != "report":
        return submit_report_response(
            state,
            tabs_update=gr.update(selected="report"),
            error_message_update="You must complete the interview before submitting a report.",
            submit_report_confirm_update=gr.update(visible=False)
        )

    state["phase"] = "grading"

    scenario = state["scenario"]
    checklist = state["checklist"]
    coverage_flags = state["coverage_flags"]

    progress(0.05, desc="Grading report...")
    grading_results = grade_report(
        narrative,
        scenario,
        checklist,
        coverage_flags,
        progress_callback=progress
    )

    progress(0.90, desc="Saving session...")

    session_id, json_path, pdf_path = save_session(
        scenario,
        checklist,
        state["history"],
        coverage_flags,
        narrative,
        grading_results["narrative_coverage"],
        grading_results
    )

    progress(0.96, desc="Formatting results...")
    grading_display = format_grading_display(grading_results)

    state["phase"] = "complete"

    progress(1, desc="Done!")

    return submit_report_response(
        state,
        pdf_download_update=gr.update(value=pdf_path, visible=True),
        tabs_update=gr.update(selected="grading"),
        grading_display_update=grading_display,
        error_message_update="",
        interview_tab_update=gr.update(visible=True),
        report_tab_update=gr.update(visible=True),
        grading_tab_update=gr.update(visible=True),
        message_input_update=gr.update(interactive=False),
        send_btn_update=gr.update(interactive=False),
        end_interview_btn_update=gr.update(interactive=False),
        narrative_input_update=gr.update(interactive=False),
        submit_report_btn_update=gr.update(interactive=False),
        submit_report_confirm_update=gr.update(visible=False)
    )

In [ ]:
def format_grading_display(grading_results):
    def esc(value):
        if value is None:
            return "—"
        return html_utils.escape(str(value), quote=True)

    def truncate(value, max_len=50):
        value = "" if value is None else str(value)
        return value[:max_len] + ("..." if len(value) > max_len else "")

    content = grading_results["content"]
    contradictions_interview = grading_results["contradictions_interview"]
    contradictions_narrative = grading_results["contradictions_narrative"]
    grammar = grading_results["grammar"]
    style = grading_results["style"]

    total = len(content)
    full_interview = sum(1 for r in content if r["asked_in_interview"] == "full")
    partial_interview = sum(1 for r in content if r["asked_in_interview"] == "partial")
    full_narrative = sum(1 for r in content if r["in_narrative"] == "full")
    partial_narrative = sum(1 for r in content if r["in_narrative"] == "partial")
    total_contradictions = len(contradictions_interview) + len(contradictions_narrative)

    html_output = f"""
<div style="font-family: sans-serif; font-size: 13px; line-height: 1.6;">

<div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 20px;">
    <div style="background: #f5f5f5; padding: 12px; border-radius: 8px;">
        <div style="font-size: 11px; color: #888;">Elements fully asked</div>
        <div style="font-size: 22px; font-weight: 500;">{full_interview} / {total}</div>
    </div>
    <div style="background: #f5f5f5; padding: 12px; border-radius: 8px;">
        <div style="font-size: 11px; color: #888;">Elements in narrative</div>
        <div style="font-size: 22px; font-weight: 500;">{full_narrative} / {total}</div>
    </div>
    <div style="background: #f5f5f5; padding: 12px; border-radius: 8px;">
        <div style="font-size: 11px; color: #888;">Contradictions</div>
        <div style="font-size: 22px; font-weight: 500; color: {'#c0392b' if total_contradictions > 0 else 'inherit'};">{total_contradictions}</div>
    </div>
    <div style="background: #f5f5f5; padding: 12px; border-radius: 8px;">
        <div style="font-size: 11px; color: #888;">Grammar & style errors</div>
        <div style="font-size: 22px; font-weight: 500;">{len(grammar) + len(style)}</div>
    </div>
</div>

<h3 style="margin-bottom: 8px;">Section 1 — Content coverage</h3>
<table style="width: 100%; border-collapse: collapse; font-size: 12px; margin-bottom: 20px;">
    <thead>
        <tr style="background: #1a1a2e; color: white;">
            <th style="padding: 6px 8px; text-align: left; color: white;">Element ID</th>
            <th style="padding: 6px 8px; text-align: left; color: white;">Element</th>
            <th style="padding: 6px 8px; text-align: left; color: white;">Asked in interview</th>
            <th style="padding: 6px 8px; text-align: left; color: white;">Interview context</th>
            <th style="padding: 6px 8px; text-align: left; color: white;">In narrative</th>
            <th style="padding: 6px 8px; text-align: left; color: white;">Narrative ref</th>
            <th style="padding: 6px 8px; text-align: left; color: white;">Notes</th>
        </tr>
    </thead>
    <tbody>"""

    for i, row in enumerate(content):
        bg = "#ffffff" if i % 2 == 0 else "#f9f9f9"

        interview_status = row["asked_in_interview"]
        interview_source_label = row.get("interview_source_label")

        if interview_status == "full":
            label_text = "Full"
            if interview_source_label:
                label_text += f" — {interview_source_label}"
            interview_label = f"<span style='background:#EAF3DE;color:#27500A;padding:2px 7px;border-radius:99px;font-size:11px;'>{esc(label_text)}</span>"
        elif interview_status == "partial":
            label_text = "Partial"
            if interview_source_label:
                label_text += f" — {interview_source_label}"
            interview_label = f"<span style='background:#FAEEDA;color:#633806;padding:2px 7px;border-radius:99px;font-size:11px;'>{esc(label_text)}</span>"
        else:
            interview_label = "<span style='background:#FCEBEB;color:#791F1F;padding:2px 7px;border-radius:99px;font-size:11px;'>Not asked</span>"

        narrative_status = row["in_narrative"]
        if narrative_status == "full":
            narrative_label = "<span style='background:#EAF3DE;color:#27500A;padding:2px 7px;border-radius:99px;font-size:11px;'>Full</span>"
        elif narrative_status == "partial":
            narrative_label = "<span style='background:#FAEEDA;color:#633806;padding:2px 7px;border-radius:99px;font-size:11px;'>Partial</span>"
        else:
            narrative_label = "<span style='background:#FCEBEB;color:#791F1F;padding:2px 7px;border-radius:99px;font-size:11px;'>Missing</span>"

        if interview_status in ("full", "partial") and row["interview_messages"]:
            if row.get("interview_source") == "interviewee_response":
                interview_ref = "<br>".join([
                    f"Line {esc(line)} response: &quot;{esc(truncate(msg.get('provided_fact') or msg.get('interviewee_response', '')))}&quot;"
                    for line, msg in row["interview_messages"].items()
                ])
            else:
                interview_ref = "<br>".join([
                    f"Line {esc(line)} question: &quot;{esc(truncate(msg.get('officer_message', '')))}&quot;"
                    for line, msg in row["interview_messages"].items()
                ])
        else:
            interview_ref = "—"

        if narrative_status in ("full", "partial") and row["narrative_phrases"]:
            narrative_ref = "<br>".join([
                f"Chunk {esc(chunk)}: &quot;{esc(truncate(phrase))}&quot;"
                for chunk, phrase in row["narrative_phrases"].items()
            ])
        else:
            narrative_ref = "—"

        notes = esc(row["notes"]) if row["notes"] else "—"

        html_output += f"""
        <tr style="background:{bg};">
            <td style="padding:6px 8px;font-family:monospace;font-size:11px;">{esc(row['element_id'])}</td>
            <td style="padding:6px 8px;">{esc(row['element'])}</td>
            <td style="padding:6px 8px;">{interview_label}</td>
            <td style="padding:6px 8px;font-size:11px;color:#555;">{interview_ref}</td>
            <td style="padding:6px 8px;">{narrative_label}</td>
            <td style="padding:6px 8px;font-size:11px;color:#555;">{narrative_ref}</td>
            <td style="padding:6px 8px;font-size:11px;color:#555;">{notes}</td>
        </tr>"""

    html_output += """
    </tbody>
</table>

<h3 style="margin-bottom: 8px;">Section 2 — Grammar errors</h3>"""

    if grammar:
        html_output += """<table style="width:100%;border-collapse:collapse;font-size:12px;margin-bottom:20px;">
        <thead><tr style="background:#1a1a2e;color:white;">
            <th style="padding:6px 8px;text-align:left;color:white;">Chunk</th>
            <th style="padding:6px 8px;text-align:left;color:white;">Source</th>
            <th style="padding:6px 8px;text-align:left;color:white;">Error type</th>
            <th style="padding:6px 8px;text-align:left;color:white;">Excerpt</th>
            <th style="padding:6px 8px;text-align:left;color:white;">Suggestion</th>
        </tr></thead><tbody>"""

        for i, err in enumerate(grammar):
            bg = "#ffffff" if i % 2 == 0 else "#f9f9f9"

            html_output += f"""<tr style="background:{bg};">
                <td style="padding:6px 8px;">{esc(err.get('chunk', '—'))}</td>
                <td style="padding:6px 8px;">{esc(err.get('source', '—'))}</td>
                <td style="padding:6px 8px;">{esc(err.get('error_type', '—'))}</td>
                <td style="padding:6px 8px;font-style:italic;">&quot;{esc(err.get('excerpt', '—'))}&quot;</td>
                <td style="padding:6px 8px;">{esc(err.get('suggestion', '—'))}</td>
            </tr>"""

        html_output += "</tbody></table>"
    else:
        html_output += "<p style='color:#3B6D11;margin-bottom:20px;'>No grammar errors found.</p>"

    html_output += "<h3 style='margin-bottom:8px;'>Section 3 — Style violations</h3>"

    if style:
        html_output += """<table style="width:100%;border-collapse:collapse;font-size:12px;margin-bottom:20px;">
        <thead><tr style="background:#1a1a2e;color:white;">
            <th style="padding:6px 8px;text-align:left;color:white;">Chunk</th>
            <th style="padding:6px 8px;text-align:left;color:white;">Rule</th>
            <th style="padding:6px 8px;text-align:left;color:white;">Error type</th>
            <th style="padding:6px 8px;text-align:left;color:white;">Excerpt</th>
            <th style="padding:6px 8px;text-align:left;color:white;">Suggestion</th>
        </tr></thead><tbody>"""

        for i, err in enumerate(style):
            bg = "#ffffff" if i % 2 == 0 else "#f9f9f9"

            html_output += f"""<tr style="background:{bg};">
                <td style="padding:6px 8px;">{esc(err.get('chunk', '—'))}</td>
                <td style="padding:6px 8px;">{esc(err.get('rule', '—'))}</td>
                <td style="padding:6px 8px;">{esc(err.get('error_type', '—'))}</td>
                <td style="padding:6px 8px;font-style:italic;">&quot;{esc(err.get('excerpt', '—'))}&quot;</td>
                <td style="padding:6px 8px;">{esc(err.get('suggestion', '—'))}</td>
            </tr>"""

        html_output += "</tbody></table>"
    else:
        html_output += "<p style='color:#3B6D11;margin-bottom:20px;'>No style violations found.</p>"

    html_output += "</div>"
    return html_output

In [ ]:
def handle_reset():
    return (
        create_session_state(),              # state
        [],                                  # chatbot
        gr.update(value="", interactive=True),  # narrative input
        "",                                  # grading display
        gr.update(visible=False),            # pdf download
        gr.update(value="", visible=False),  # dispatch bar
        gr.update(value="", visible=False),  # session info
        gr.update(visible=True),             # crime type selector
        gr.update(visible=True),             # start button
        gr.update(visible=False),            # end session button
        gr.update(visible=False),            # end session confirm row
        gr.update(selected="interview"),     # tabs back to interview
        gr.update(visible=True),             # interview tab
        gr.update(visible=False),            # report tab
        gr.update(visible=False),            # grading tab
        gr.update(interactive=True),         # message input
        gr.update(interactive=True),         # send button
        gr.update(interactive=True),         # end interview button
        gr.update(interactive=True),         # submit report button
        gr.update(visible=False),            # end interview confirm row
        gr.update(visible=False),            # submit report confirm row
        ""                                   # error message
    )

In [ ]:
def build_ui():
    with gr.Blocks(title="Police Report Training Bot") as demo:

        state = gr.State(create_session_state())

        # ── Header ────────────────────────────────────────────────────
        with gr.Row():
            gr.Markdown("## Police Report Training Bot")

        error_message = gr.Markdown("", visible=True)

        # ── Main layout ───────────────────────────────────────────────
        with gr.Row():

            # ── Sidebar ───────────────────────────────────────────────
            with gr.Column(scale=1):
                crime_type_selector = gr.CheckboxGroup(
                    choices=CRIME_TYPES,
                    value=CRIME_TYPES,
                    label="Crime type"
                )
                start_btn = gr.Button("Start session", variant="primary")
                session_info = gr.Markdown("", visible=False)

                with gr.Row(visible=False) as end_session_confirm:
                    confirm_end_btn = gr.Button("Yes, end session", variant="stop")
                    cancel_end_btn = gr.Button("Cancel")

                end_session_btn = gr.Button("End session", visible=False, variant="secondary")

            # ── Content area ──────────────────────────────────────────
            with gr.Column(scale=3):
                dispatch_bar = gr.Markdown("", visible=False)

                with gr.Tabs(selected="interview") as tabs:

                    # ── Tab 1 — Interview ─────────────────────────────
                    with gr.Tab("Interview", id="interview") as interview_tab:
                        chatbot = gr.Chatbot(
                            label="Interview",
                            height=400,
                            type="messages"
                        )
                        with gr.Row():
                            message_input = gr.Textbox(
                                placeholder="Ask your next question...",
                                show_label=False,
                                scale=4
                            )
                            send_btn = gr.Button("Send", scale=1, variant="primary")
                        end_interview_btn = gr.Button("End interview", variant="secondary")

                        with gr.Column(visible=False) as end_interview_confirm:
                            gr.Markdown(
                                "Are you sure you are done with the interview? "
                                "Once you start writing the report, you will not be able to access or add to the interview until grading is complete."
                            )
                            with gr.Row():
                                confirm_end_interview_btn = gr.Button("Yes, start report", variant="stop")
                                cancel_end_interview_btn = gr.Button("Cancel")

                    # ── Tab 2 — Write report ──────────────────────────
                    with gr.Tab("Write report", id="report", visible=False) as report_tab:
                        gr.Markdown("Write your narrative report below. Include all relevant facts from the interview.")
                        narrative_input = gr.Textbox(
                            placeholder="Write your report here...",
                            lines=20,
                            show_label=False
                        )
                        submit_report_btn = gr.Button("Submit report", variant="primary")

                        with gr.Column(visible=False) as submit_report_confirm:
                            gr.Markdown(
                                "Are you sure you are done with the report? "
                                "Once grading starts, you will not be able to edit or resubmit the report."
                            )
                            with gr.Row():
                                confirm_submit_report_btn = gr.Button("Yes, grade report", variant="stop")
                                cancel_submit_report_btn = gr.Button("Cancel")

                    # ── Tab 3 — Grading ───────────────────────────────
                    with gr.Tab("Grading", id="grading", visible=False) as grading_tab:
                        grading_display = gr.HTML("")
                        pdf_download = gr.File(label="Download grading report", visible=False)

        # ── Event wiring ──────────────────────────────────────────────

        start_btn.click(
            fn=handle_start,
            inputs=[crime_type_selector],
            outputs=[
                dispatch_bar,
                chatbot,
                state,
                crime_type_selector,
                start_btn,
                session_info,
                end_session_btn,
                tabs,
                interview_tab,
                report_tab,
                grading_tab,
                message_input,
                send_btn,
                end_interview_btn,
                error_message
            ]
        )

        send_btn.click(
            fn=handle_chat,
            inputs=[message_input, chatbot, state],
            outputs=[chatbot, message_input, state]
        )

        message_input.submit(
            fn=handle_chat,
            inputs=[message_input, chatbot, state],
            outputs=[chatbot, message_input, state]
        )

        end_interview_btn.click(
            fn=handle_request_end_interview,
            inputs=[state],
            outputs=[state, end_interview_confirm, error_message]
        )

        cancel_end_interview_btn.click(
            fn=handle_cancel_end_interview,
            outputs=[end_interview_confirm, error_message]
        )

        confirm_end_interview_btn.click(
            fn=handle_confirm_end_interview,
            inputs=[state],
            outputs=[
                state,
                tabs,
                interview_tab,
                report_tab,
                grading_tab,
                message_input,
                send_btn,
                end_interview_btn,
                end_interview_confirm,
                narrative_input,
                submit_report_btn,
                error_message
            ]
        )

        submit_report_btn.click(
            fn=handle_request_submit_report,
            inputs=[narrative_input, state],
            outputs=[submit_report_confirm, error_message]
        )

        cancel_submit_report_btn.click(
            fn=handle_cancel_submit_report,
            outputs=[submit_report_confirm, error_message]
        )

        confirm_submit_report_btn.click(
            fn=handle_submit_report,
            inputs=[narrative_input, state],
            outputs=[
                state,
                pdf_download,
                tabs,
                grading_display,
                error_message,
                interview_tab,
                report_tab,
                grading_tab,
                message_input,
                send_btn,
                end_interview_btn,
                narrative_input,
                submit_report_btn,
                submit_report_confirm
            ]
        )

        end_session_btn.click(
            fn=lambda: gr.update(visible=True),
            outputs=[end_session_confirm]
        )

        cancel_end_btn.click(
            fn=lambda: gr.update(visible=False),
            outputs=[end_session_confirm]
        )

        confirm_end_btn.click(
            fn=handle_reset,
            inputs=[],
            outputs=[
                state,
                chatbot,
                narrative_input,
                grading_display,
                pdf_download,
                dispatch_bar,
                session_info,
                crime_type_selector,
                start_btn,
                end_session_btn,
                end_session_confirm,
                tabs,
                interview_tab,
                report_tab,
                grading_tab,
                message_input,
                send_btn,
                end_interview_btn,
                submit_report_btn,
                end_interview_confirm,
                submit_report_confirm,
                error_message
            ]
        )

    return demo

In [ ]:
def launch():
    demo = build_ui()
    demo.queue()
    demo.launch(share=True, debug=True)